In [39]:
!pip -q install gradio openpyxl pandas

In [40]:
import pandas as pd
import gradio as gr
import re

from google.colab import files

import warnings
warnings.filterwarnings("ignore")

In [41]:
uploaded = files.upload()

Saving tech_financials_2023_2025.xlsx to tech_financials_2023_2025 (4).xlsx


In [42]:
filename = list(uploaded.keys())[0]

raw_df = pd.read_excel(filename)

print(raw_df.head())

  Tech Giants Financial Data (FY2023–FY2025) — Extracted from SEC 10-K Filings  \
0                                            Company                             
1                                          Microsoft                             
2                                          Microsoft                             
3                                          Microsoft                             
4                                              Tesla                             

    Unnamed: 1       Unnamed: 2          Unnamed: 3       Unnamed: 4  \
0  Fiscal Year  Fiscal Year End  Total Revenue ($M)  Net Income ($M)   
1       FY2023     Jun 30, 2023              211915            72361   
2       FY2024     Jun 30, 2024              245122            88136   
3       FY2025     Jun 30, 2025              281739           101821   
4       FY2023     Dec 31, 2023               96773            14999   

          Unnamed: 5              Unnamed: 6               Unnamed: 7  \
0

In [43]:
df = pd.read_excel(filename, header=1)

df = df[[
    "Company",
    "Fiscal Year",
    "Total Revenue ($M)",
    "Net Income ($M)",
    "Total Assets ($M)",
    "Total Liabilities ($M)",
    "CF from Operations ($M)"
]]

df.columns = [
    "Company",
    "Year",
    "Revenue",
    "Net Income",
    "Assets",
    "Liabilities",
    "Operating Cash Flow"
]

In [44]:
numeric_cols = [
    "Revenue",
    "Net Income",
    "Assets",
    "Liabilities",
    "Operating Cash Flow"
]

for col in numeric_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df["Year"] = df["Year"].astype(str).str.replace('FY', '').astype(int)

df = df.sort_values(
    ["Company", "Year"]
).reset_index(drop=True)

In [45]:
print("="*60)

print("Dataset Loaded Successfully")

print("="*60)

print(df.info())

print()

print(df.head())

print()

print("Companies")

print(df["Company"].unique())

print()

print("Years")

print(sorted(df["Year"].unique()))

print("="*60)

Dataset Loaded Successfully
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Company              9 non-null      object
 1   Year                 9 non-null      int64 
 2   Revenue              9 non-null      int64 
 3   Net Income           9 non-null      int64 
 4   Assets               9 non-null      int64 
 5   Liabilities          9 non-null      int64 
 6   Operating Cash Flow  9 non-null      int64 
dtypes: int64(6), object(1)
memory usage: 636.0+ bytes
None

     Company  Year  Revenue  Net Income  Assets  Liabilities  \
0      Apple  2023   383285       96995  352583       290437   
1      Apple  2024   391035       93736  364980       308034   
2      Apple  2025   416161      112010  399200       346000   
3  Microsoft  2023   211915       72361  411976       205753   
4  Microsoft  2024   245122       88136  512163   

In [46]:
SUPPORTED_COMPANIES = sorted(
    df["Company"].unique().tolist()
)

SUPPORTED_METRICS = [
    "Revenue",
    "Net Income",
    "Assets",
    "Liabilities",
    "Operating Cash Flow"
]

LATEST_YEAR = int(df["Year"].max())

In [47]:
df

,Company,Year,Revenue,Net Income,Assets,Liabilities,Operating Cash Flow
0,Apple,2023,383285,96995,352583,290437,110543
1,Apple,2024,391035,93736,364980,308034,118254
2,Apple,2025,416161,112010,399200,346000,136000
3,Microsoft,2023,211915,72361,411976,205753,87582
4,Microsoft,2024,245122,88136,512163,243686,118548
5,Microsoft,2025,281739,101821,619003,275524,136257
6,Tesla,2023,96773,14999,106618,43009,13256
7,Tesla,2024,97690,7093,122070,52773,14923
8,Tesla,2025,94822,-1100,132000,57000,11500


In [48]:
print(df.columns.tolist())

['Company', 'Year', 'Revenue', 'Net Income', 'Assets', 'Liabilities', 'Operating Cash Flow']


In [49]:
print(df.shape)

(9, 7)


In [50]:
print(df.dtypes)

Company                object
Year                    int64
Revenue                 int64
Net Income              int64
Assets                  int64
Liabilities             int64
Operating Cash Flow     int64
dtype: object


In [51]:
financial_definitions = {
    "Revenue":
        "Revenue is the total income generated from a company's core business operations before deducting expenses.",

    "Net Income":
        "Net Income is the company's profit after subtracting all operating costs, taxes, and other expenses.",

    "Assets":
        "Assets are resources owned by a company that have economic value and can generate future benefits.",

    "Liabilities":
        "Liabilities are financial obligations or debts that a company owes to others.",

    "Operating Cash Flow":
        "Operating Cash Flow represents the cash generated from a company's normal business operations."
}

In [52]:
def extract_company(query):

    query = query.lower()

    aliases = {

        "Microsoft": [
            "microsoft",
            "msft"
        ],

        "Apple": [
            "apple",
            "aapl"
        ],

        "Tesla": [
            "tesla",
            "tsla"
        ]
    }

    companies = []

    for company, names in aliases.items():

        for name in names:

            if re.search(r"\b" + re.escape(name) + r"\b", query):

                companies.append(company)
                break

    return companies

In [53]:
def extract_metric(query):

    query = query.lower()

    metric_map = {

        "Revenue": [
            "revenue",
            "sales",
            "turnover",
            "top line"
        ],

        "Net Income": [
            "net income",
            "profit",
            "profits",
            "earnings",
            "net profit",
            "bottom line"
        ],

        "Assets": [
            "assets",
            "asset"
        ],

        "Liabilities": [
            "liabilities",
            "liability",
            "debt",
            "debts",
            "obligations"
        ],

        "Operating Cash Flow": [
            "operating cash flow",
            "cash flow",
            "cash generated",
            "cash from operations",
            "ocf"
        ]

    }

    for metric, words in metric_map.items():

        for word in sorted(words, key=len, reverse=True):

            if re.search(r"\b" + re.escape(word) + r"\b", query):

                return metric

    return None

In [54]:
def extract_year(query):

    years = re.findall(r"\b20\d{2}\b", query)

    if years:

        year = int(years[0])

        if year in df["Year"].unique():

            return year

    return LATEST_YEAR

In [55]:
def detect_intent(query):

    q = query.lower()

    if any(x in q for x in ["what is", "define", "meaning", "explain"]):
        return "definition"

    if any(x in q for x in ["compare", "comparison", "versus", "vs"]):
        return "compare"

    if any(x in q for x in ["highest", "largest", "maximum", "top"]):
        return "highest"

    if any(x in q for x in ["lowest", "smallest", "minimum", "least"]):
        return "lowest"

    if any(x in q for x in ["trend", "growth", "over years", "history"]):
        return "trend"

    if any(x in q for x in ["report", "statement"]):
        return "report"

    if any(x in q for x in ["summary", "summarize"]):
        return "summary"

    if any(x in q for x in ["insight", "analysis", "performance", "health"]):
        return "insight"

    if any(x in q for x in ["help", "what can you do"]):
        return "help"

    return "data"

In [56]:
def validate_query(query):

    intent = detect_intent(query)

    companies = extract_company(query)

    metric = extract_metric(query)

    if intent in ["definition", "help"]:
        return True, ""

    if intent in ["highest", "lowest"]:

        if metric:

            return True, ""

        return False, "Please specify a financial metric."

    if intent in ["compare"]:

        if metric:

            return True, ""

        return False, "Please specify a metric to compare."

    if len(companies) == 0:

        return False, (
            "I currently support only Microsoft, Apple, and Tesla."
        )

    return True, ""

In [57]:
def parse_query(query):

    return {

        "intent": detect_intent(query),

        "companies": extract_company(query),

        "metric": extract_metric(query),

        "year": extract_year(query)
    }

In [58]:
tests = [

    "Tell me Apple's revenue",

    "Microsoft report",

    "Tesla assets",

    "Compare Microsoft and Apple revenue",

    "Highest revenue",

    "Lowest liabilities",

    "Tesla revenue trend",

    "Apple financial summary",

    "Financial health of Microsoft",

    "Revenue in 2024",

    "What is net income?",

    "Explain operating cash flow",

    "Amazon revenue",

    "Help"

]

for question in tests:

    print("="*70)

    print(question)

    print(parse_query(question))

    print(validate_query(question))

Tell me Apple's revenue
{'intent': 'data', 'companies': ['Apple'], 'metric': 'Revenue', 'year': 2025}
(True, '')
Microsoft report
{'intent': 'report', 'companies': ['Microsoft'], 'metric': None, 'year': 2025}
(True, '')
Tesla assets
{'intent': 'data', 'companies': ['Tesla'], 'metric': 'Assets', 'year': 2025}
(True, '')
Compare Microsoft and Apple revenue
{'intent': 'compare', 'companies': ['Microsoft', 'Apple'], 'metric': 'Revenue', 'year': 2025}
(True, '')
Highest revenue
{'intent': 'highest', 'companies': [], 'metric': 'Revenue', 'year': 2025}
(True, '')
Lowest liabilities
{'intent': 'lowest', 'companies': [], 'metric': 'Liabilities', 'year': 2025}
(True, '')
Tesla revenue trend
{'intent': 'trend', 'companies': ['Tesla'], 'metric': 'Revenue', 'year': 2025}
(True, '')
Apple financial summary
{'intent': 'summary', 'companies': ['Apple'], 'metric': None, 'year': 2025}
(True, '')
Financial health of Microsoft
{'intent': 'insight', 'companies': ['Microsoft'], 'metric': None, 'year': 2025}

In [59]:
def get_company_data(company, year=LATEST_YEAR):

    data = df[
        (df["Company"] == company) &
        (df["Year"] == year)
    ]

    if data.empty:
        return None

    return data.iloc[0]

In [60]:
def format_money(value):

    if pd.isna(value):
        return "N/A"

    return f"${value/1000:.2f} Billion"

In [61]:
def company_metric(company, metric, year):

    row = get_company_data(company, year)

    if row is None:
        return "No data available."

    return f"""
📊 {company} ({year})

{metric}

{format_money(row[metric])}
"""

In [62]:
def company_report(company, year):

    row = get_company_data(company, year)

    if row is None:
        return "No report available."

    return f"""
# 📄 {company} Financial Report ({year})

Revenue
{format_money(row["Revenue"])}

Net Income
{format_money(row["Net Income"])}

Assets
{format_money(row["Assets"])}

Liabilities
{format_money(row["Liabilities"])}

Operating Cash Flow
{format_money(row["Operating Cash Flow"])}
"""

In [63]:
def financial_insight(company, year):

    row = get_company_data(company, year)

    if row is None:
        return "No data available."

    insights = []

    if row["Revenue"] > 200000:
        insights.append("✅ Strong revenue generation.")

    if row["Net Income"] > 50000:
        insights.append("✅ High profitability.")

    if row["Assets"] > row["Liabilities"]:
        insights.append("✅ Assets exceed liabilities.")

    if row["Operating Cash Flow"] > row["Net Income"]:
        insights.append("✅ Strong operating cash flow.")

    if not insights:
        insights.append("Financial indicators are mixed.")

    return (
        f"## 📈 {company} Financial Insight ({year})\n\n"
        + "\n".join(insights)
    )

In [64]:
def highest_metric(metric, year):

    data = df[df["Year"] == year]

    row = data.loc[data[metric].idxmax()]

    return f"""
🏆 Highest {metric}

Company

{row['Company']}

Value

{format_money(row[metric])}
"""

In [65]:
def lowest_metric(metric, year):

    data = df[df["Year"] == year]

    row = data.loc[data[metric].idxmin()]

    return f"""
📉 Lowest {metric}

Company

{row['Company']}

Value

{format_money(row[metric])}
"""

In [66]:
def compare_metric(metric, year):

    data = (
        df[df["Year"] == year]
        .sort_values(metric, ascending=False)
    )

    text = f"## 📊 {metric} Comparison ({year})\n\n"

    medals = ["🥇", "🥈", "🥉"]

    for i, (_, row) in enumerate(data.iterrows()):

        medal = medals[i] if i < 3 else "•"

        text += (
            f"{medal} {row['Company']}\n"
            f"{format_money(row[metric])}\n\n"
        )

    leader = data.iloc[0]["Company"]

    text += f"Leader: {leader}"

    return text

In [67]:
def trend_summary(company, metric):

    data = (
        df[df["Company"] == company]
        .sort_values("Year")
    )

    lines = []

    for _, row in data.iterrows():

        lines.append(
            f"{row['Year']} → {format_money(row[metric])}"
        )

    start = data.iloc[0][metric]
    end = data.iloc[-1][metric]

    if start == 0:
        change = 0
    else:
        change = ((end - start) / start) * 100

    direction = "increased" if change >= 0 else "decreased"

    return (
        f"📈 {company} {metric} Trend\n\n"
        + "\n".join(lines)
        + f"\n\nOverall, {metric.lower()} has {direction} by {abs(change):.2f}% over the available period."
    )

In [68]:
print(company_metric("Apple", "Revenue", 2025))

print(company_report("Microsoft", 2025))

print(financial_insight("Tesla", 2025))

print(highest_metric("Revenue", 2025))

print(lowest_metric("Liabilities", 2025))

print(compare_metric("Revenue", 2025))

print(trend_summary("Apple", "Revenue"))


📊 Apple (2025)

Revenue

$416.16 Billion


# 📄 Microsoft Financial Report (2025)

Revenue
$281.74 Billion

Net Income
$101.82 Billion

Assets
$619.00 Billion

Liabilities
$275.52 Billion

Operating Cash Flow
$136.26 Billion

## 📈 Tesla Financial Insight (2025)

✅ Assets exceed liabilities.
✅ Strong operating cash flow.

🏆 Highest Revenue

Company

Apple

Value

$416.16 Billion


📉 Lowest Liabilities

Company

Tesla

Value

$57.00 Billion

## 📊 Revenue Comparison (2025)

🥇 Apple
$416.16 Billion

🥈 Microsoft
$281.74 Billion

🥉 Tesla
$94.82 Billion

Leader: Apple
📈 Apple Revenue Trend

2023 → $383.29 Billion
2024 → $391.04 Billion
2025 → $416.16 Billion

Overall, revenue has increased by 8.58% over the available period.


In [69]:
HELP_TEXT = """
# 💹 Financial Analysis Chatbot

I currently support financial analysis for:

• Microsoft
• Apple
• Tesla

--------------------------------------------

📊 Financial Data

• Tell me Apple's revenue

• Microsoft assets

• Tesla liabilities

--------------------------------------------

📄 Reports

• Apple report

• Tesla financial report

• Microsoft report

--------------------------------------------

📈 Trends

• Apple revenue trend

• Tesla operating cash flow trend

--------------------------------------------

📊 Comparisons

• Compare Microsoft and Apple revenue

• Compare operating cash flow

• Compare net income

--------------------------------------------

🏆 Rankings

• Highest revenue

• Lowest liabilities

• Highest assets

--------------------------------------------

📖 Definitions

• What is revenue?

• Explain operating cash flow

• Define liabilities
"""

In [77]:
def financial_definition(metric):

    if metric is None:

        return """
Please specify a financial term.

Examples

• What is revenue?

• Explain operating cash flow

• Define liabilities
"""

    return financial_definitions.get(
        metric,
        "Definition not available."
    )

In [78]:
def chatbot_response(query):

    valid, message = validate_query(query)

    if not valid:
        return message

    parsed = parse_query(query)

    intent = parsed["intent"]
    companies = parsed["companies"]
    metric = parsed["metric"]
    year = parsed["year"]

    # Help
    if intent == "help":
        return HELP_TEXT

    if intent == "definition":
        return financial_definition(metric)

    # Highest
    if intent == "highest":
        return highest_metric(metric, year)

    # Lowest
    if intent == "lowest":
        return lowest_metric(metric, year)

    # Comparison
    if intent == "compare":
        return compare_metric(metric, year)

    # Report
    if intent == "report":
        return company_report(companies[0], year)

    # Summary / Insight
    if intent in ["summary", "insight"]:
        return financial_insight(companies[0], year)

    # Trend
    if intent == "trend":
        return trend_summary(companies[0], metric)

    # Default → Company Metric
    return company_metric(companies[0], metric, year)

In [79]:
questions = [

"Tell me Apple's revenue",
"Tell me Microsoft's net income",
"Tesla liabilities",
"Apple assets",

"Compare Microsoft and Apple revenue",
"Compare net income",
"Compare operating cash flow",

"Highest revenue",
"Lowest liabilities",
"Highest assets",

"Apple report",
"Tesla financial report",
"Microsoft report",

"Apple financial health",
"Tesla summary",
"Microsoft insight",

"Apple revenue trend",
"Tesla operating cash flow trend",

"What is revenue?",
"What is net income?",
"Explain liabilities",

"Help",

"Amazon revenue",

"Compare",

"Report"

]

for q in questions:

    print("="*70)
    print(q)
    print()
    print(chatbot_response(q))
    print()

Tell me Apple's revenue


📊 Apple (2025)

Revenue

$416.16 Billion


Tell me Microsoft's net income


📊 Microsoft (2025)

Net Income

$101.82 Billion


Tesla liabilities


📊 Tesla (2025)

Liabilities

$57.00 Billion


Apple assets


📊 Apple (2025)

Assets

$399.20 Billion


Compare Microsoft and Apple revenue

## 📊 Revenue Comparison (2025)

🥇 Apple
$416.16 Billion

🥈 Microsoft
$281.74 Billion

🥉 Tesla
$94.82 Billion

Leader: Apple

Compare net income

## 📊 Net Income Comparison (2025)

🥇 Apple
$112.01 Billion

🥈 Microsoft
$101.82 Billion

🥉 Tesla
$-1.10 Billion

Leader: Apple

Compare operating cash flow

## 📊 Operating Cash Flow Comparison (2025)

🥇 Microsoft
$136.26 Billion

🥈 Apple
$136.00 Billion

🥉 Tesla
$11.50 Billion

Leader: Microsoft

Highest revenue


🏆 Highest Revenue

Company

Apple

Value

$416.16 Billion


Lowest liabilities


📉 Lowest Liabilities

Company

Tesla

Value

$57.00 Billion


Highest assets


🏆 Highest Assets

Company

Microsoft

Value

$619.00 Billion


Appl

In [80]:
def financial_chat(message, history):

    return chatbot_response(message)

In [81]:
demo = gr.ChatInterface(

    fn=financial_chat,

    title="💹 Financial Analysis Chatbot",

    description="""
Analyze financial statements of Microsoft, Apple and Tesla (2023–2025).

Features

• Financial Reports

• Company Comparison

• Revenue Analysis

• Financial Insights

• Trend Analysis

• Financial Definitions
""",

    examples=[

        "Tell me Apple's revenue",

        "Compare Microsoft and Apple revenue",

        "Highest revenue",

        "Apple report",

        "Tesla financial health",

        "Apple revenue trend",

        "What is operating cash flow?",

        "Help"

    ],

    theme="soft"
)

In [82]:
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://28c2638eef9504cd0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://28c2638eef9504cd0e.gradio.live
